# Criação da Fato — Assistência Estudantil

Rode o notebook `02_construcao_dimensoes.ipynb` antes deste (gera/valida os CSVs
de dimensão usados na conferência final).

Usa a função `criar_fato_assistencia` já corrigida:
- filtro do Campus I com a lista `cursos_fora` (mesma da dimensão Curso);
- turno nulo -> `ID_TURNO = 0` ("Não informado"), sem descartar linha;
- **merge único** com `dim_curso_centro` para trazer `ID_CENTRO`
  (antes havia um merge duplicado que quebrava essa coluna em
  `ID_CENTRO_x` / `ID_CENTRO_y`);
- validação automática: se sobrar `ID_CENTRO` nulo após o merge, a função
  levanta erro listando os cursos órfãos, em vez de deixar passar silencioso.

## Função `criar_fato_assistencia`

In [ ]:
import pandas as pd


def criar_fato_assistencia(
    fato: pd.DataFrame,
    curso_centro: pd.DataFrame
) -> pd.DataFrame:
    """
    Constrói a tabela fato da assistência estudantil.
    """

    # -------------------------------------------------
    # 1. Filtrar apenas cursos do Campus I
    # -------------------------------------------------
    cursos_fora = [
        13403, 13454, 13455, 13457, 80589, 97767,
        98976, 98980, 98982, 98984, 99045,
        107348, 107352, 107356, 107360,
        109626, 113709, 397767, 1161324,
        1167933, 1440696, 5000897,
        5000898, 113699, 1110415,
        113701
    ]

    fato = fato[
        ~fato["CO_CURSO"].isin(cursos_fora)
    ].copy()

    # -------------------------------------------------
    # 2. Renomear chaves (CO_CURSO vira ID_CURSO)
    # -------------------------------------------------
    fato = fato.rename(columns={
        "CO_CURSO": "ID_CURSO",
        "TP_SEXO": "ID_SEXO",
        "TP_COR_RACA": "ID_RACA",
        "TP_TURNO": "ID_TURNO"
    })

    # -------------------------------------------------
    # 3. Turno não informado
    # -------------------------------------------------
    fato["ID_TURNO"] = (
        fato["ID_TURNO"]
        .fillna(0)
        .astype(int)
    )

    # -------------------------------------------------
    # 4. Auxílios específicos
    # -------------------------------------------------
    colunas_auxilio = [
        "IN_APOIO_ALIMENTACAO",
        "IN_APOIO_MORADIA",
        "IN_APOIO_TRANSPORTE",
        "IN_APOIO_MATERIAL_DIDATICO",
        "IN_APOIO_BOLSA_PERMANENCIA",
        "IN_APOIO_BOLSA_TRABALHO"
    ]

    fato[colunas_auxilio] = (
        fato[colunas_auxilio]
        .fillna(0)
        .astype(int)
    )

    # -------------------------------------------------
    # 5. Recebe auxílio?
    # -------------------------------------------------
    # ATENÇÃO: aqui RECEBE_AUXILIO é definido apenas por IN_APOIO_SOCIAL.
    # Se IN_APOIO_SOCIAL não for o agregador oficial de todos os auxílios,
    # considere usar: fato[colunas_auxilio + ["IN_APOIO_SOCIAL"]].any(axis=1)
    fato["RECEBE_AUXILIO"] = (
        fato["IN_APOIO_SOCIAL"]
        .fillna(0)
        .astype(int)
    )

    # -------------------------------------------------
    # 6. Indicador de Demanda Potencial Não Atendida
    # -------------------------------------------------
    fato["IDPNA"] = (
        (fato["IN_ACAO_AFIRMATIVA"] == 1) &
        (fato["RECEBE_AUXILIO"] == 0)
    ).astype(int)

    # -------------------------------------------------
    # 7. Quantidade de estudantes classificados
    # -------------------------------------------------
    fato["TOTAL_IDPNA"] = (
        fato["TOTAL_ALUNOS"] * fato["IDPNA"]
    )

    # -------------------------------------------------
    # 8. Acrescentar Centro (chave normalizada + merge único)
    # -------------------------------------------------
    fato["ID_CURSO"] = fato["ID_CURSO"].astype(int)

    curso_centro_dedup = (
        curso_centro[["ID_CURSO", "ID_CENTRO"]]
        .assign(ID_CURSO=lambda df: df["ID_CURSO"].astype(int))
        .drop_duplicates(subset="ID_CURSO")
    )

    fato = fato.merge(
        curso_centro_dedup,
        on="ID_CURSO",
        how="left",
        validate="m:1"
    )

    sem_centro = fato.loc[fato["ID_CENTRO"].isna(), "ID_CURSO"].unique()
    if len(sem_centro) > 0:
        raise ValueError(
            f"Cursos sem ID_CENTRO mapeado em dim_curso_centro: {sorted(sem_centro)}"
        )

    # -------------------------------------------------
    # 9. Organizar colunas
    # -------------------------------------------------
    fato = fato[
        [
            "ID_CURSO",
            "ID_CENTRO",
            "ID_SEXO",
            "ID_RACA",
            "ID_TURNO",
            "IN_ACAO_AFIRMATIVA",
            "IN_APOIO_SOCIAL",
            "IN_APOIO_ALIMENTACAO",
            "IN_APOIO_MORADIA",
            "IN_APOIO_TRANSPORTE",
            "IN_APOIO_MATERIAL_DIDATICO",
            "IN_APOIO_BOLSA_PERMANENCIA",
            "IN_APOIO_BOLSA_TRABALHO",
            "TOTAL_ALUNOS",
            "RECEBE_AUXILIO",
            "IDPNA",
            "TOTAL_IDPNA"
        ]
    ]

    return fato

## Carregar dados de entrada

In [ ]:
fato_raw = pd.read_csv("fato_assistencia_raw.csv")
dim_curso_centro = pd.read_csv("dim_curso_centro.csv")

print("Linhas na fato bruta:", len(fato_raw))
print("Cursos únicos na fato bruta (todos os campi):", fato_raw["CO_CURSO"].nunique())

## Por que tantos nulos/zeros nas colunas de auxílio específico?

Antes de rodar a função, vale documentar (e provar) o padrão dos dados
brutos: as colunas `IN_APOIO_ALIMENTACAO`, `IN_APOIO_MORADIA` etc. só vêm
preenchidas quando `IN_APOIO_SOCIAL = 1`. Quando `IN_APOIO_SOCIAL = 0`, o
SEDAP+ não detalha nada — o campo fica **nulo porque não se aplica**, e não
porque a informação faltou. Por isso o `fillna(0)` na função é seguro: nulo
e zero, nesse caso, significam exatamente a mesma coisa (aluno não recebeu
aquele auxílio).

In [ ]:
colunas_auxilio_check = [
    "IN_APOIO_ALIMENTACAO", "IN_APOIO_MORADIA", "IN_APOIO_TRANSPORTE",
    "IN_APOIO_MATERIAL_DIDATICO", "IN_APOIO_BOLSA_PERMANENCIA",
    "IN_APOIO_BOLSA_TRABALHO"
]

print("%% de nulos nas colunas de auxílio quando IN_APOIO_SOCIAL == 0 (esperado: 100%):")
print(
    fato_raw.loc[fato_raw["IN_APOIO_SOCIAL"] == 0, colunas_auxilio_check]
    .isnull().mean() * 100
)

print()
print("%% de nulos nas colunas de auxílio quando IN_APOIO_SOCIAL == 1 (esperado: 0%):")
print(
    fato_raw.loc[fato_raw["IN_APOIO_SOCIAL"] == 1, colunas_auxilio_check]
    .isnull().mean() * 100
)

## Gerar a Fato

In [ ]:
fato = criar_fato_assistencia(fato_raw, dim_curso_centro)
display(fato.head(10))

## Salvar Fato

In [ ]:
fato.to_csv(
    "fato_assistencia.csv",
    index=False,
    encoding="utf-8-sig"
)

## Conferir

In [ ]:
print(fato.columns.tolist())

In [ ]:
fato.isnull().sum()

In [ ]:
print("Cursos:", fato["ID_CURSO"].nunique())
print("Centros:", fato["ID_CENTRO"].nunique())

In [ ]:
print(fato["IDPNA"].value_counts())

In [ ]:
print(fato["RECEBE_AUXILIO"].value_counts())

In [ ]:
print("Total de alunos:", fato["TOTAL_ALUNOS"].sum())

In [ ]:
fato.groupby("ID_CURSO")[["TOTAL_ALUNOS", "TOTAL_IDPNA"]].sum().head(10)

In [ ]:
fato.groupby("ID_CENTRO")[["TOTAL_ALUNOS", "TOTAL_IDPNA"]].sum()

## Conferência cruzada com as dimensões

Confere se toda chave estrangeira da Fato existe na respectiva dimensão
(nenhuma linha "órfã"), lendo os CSVs gerados/validados no notebook de
dimensões.

In [ ]:
dim_curso_campus1 = pd.read_csv("/content/dim_curso_campus1.csv")
dim_sexo = pd.read_csv("/content/dim_sexo.csv")
dim_raca = pd.read_csv("/content/dim_raca.csv")
dim_turno = pd.read_csv("/content/dim_turno.csv")
dim_centro = pd.read_csv("/content/dim_centro.csv")

checagens = {
    "ID_CURSO -> dim_curso_campus1": (fato["ID_CURSO"], dim_curso_campus1["ID_CURSO"]),
    "ID_SEXO -> dim_sexo": (fato["ID_SEXO"], dim_sexo["ID_SEXO"]),
    "ID_RACA -> dim_raca": (fato["ID_RACA"], dim_raca["ID_RACA"]),
    "ID_TURNO -> dim_turno": (fato["ID_TURNO"], dim_turno["ID_TURNO"]),
    "ID_CENTRO -> dim_centro": (fato["ID_CENTRO"], dim_centro["ID_CENTRO"]),
}

for nome, (chave_fato, chave_dim) in checagens.items():
    orfaos = set(chave_fato) - set(chave_dim)
    status = "OK" if not orfaos else f"FALTANDO: {sorted(orfaos)}"
    print(f"{nome}: {status}")

In [ ]:
cursos_dim = set(dim_curso_campus1["ID_CURSO"])
cursos_fato = set(fato["ID_CURSO"])

faltando = cursos_dim - cursos_fato

print("Cursos na dimensão sem nenhuma linha na fato:", len(faltando))
print(sorted(faltando))

In [ ]:
dim_curso_campus1[
    dim_curso_campus1["ID_CURSO"].isin(faltando)
][["ID_CURSO", "CURSO"]]